# 教学演示：酒店推荐 Agent + MCP 工具调用

**本课四个重点**：
1. 理解 **MCP（Model Context Protocol）**：工具由独立 Server 暴露，Client/Agent 远程调用。
2. 对比 **直接调 MCP 工具** vs **Agent 自动决定何时调工具**。
3. 用 `langchain-mcp-adapters` 把 MCP 工具加载为 LangChain Tool，再 `create_agent` 组装 Agent。
4. 用 `astream_events` 观察：工具调用参数 → 工具返回 → 模型最终推荐文案。

> 对应脚本：`hotel_recommendation_demo_mcp.py`  
> 与 Chapter-5 的区别：工具不在 notebook 里 `@tool` 定义，而是从 `hotel_mcp_server.py` 经 **HTTP/SSE** 拉取。

## 0. 环境与前置条件

### 安装依赖

```bash
cd Chapter-7/Mcp
pip install -r requirements.txt
```

### 配置 `.env`（书根目录）

| 变量 | 用途 |
|------|------|
| `DASHSCOPE_API_KEY` | 大模型（Agent 侧） |
| `BAIDU_MAP_AK` | 百度地图 POI（**MCP Server 侧**查酒店） |

### 启动 MCP Server（必须在另一个终端先跑起来）

```bash
cd Chapter-7/Mcp
python hotel_mcp_server.py
# 默认 SSE 地址：http://127.0.0.1:8765/sse
```

### 架构一览

```
notebook / demo
    │  langchain-mcp-adapters (SSE)
    ▼
hotel_mcp_server.py
    ├── recommend_hotel_tool  →  hotel_core  →  百度/高德 API
    └── hotel_agent_query     →  LangChain Agent（本课不用）
```

In [4]:
# 路径引导 + Windows 中文输出（Jupyter 必须先跑本格）
import sys
from pathlib import Path

_candidates = [Path.cwd(), Path.cwd() / "Chapter-7" / "Mcp"]
_mcp_dir = next((p for p in _candidates if (p / "mcp_paths.py").exists()), None)
if _mcp_dir is None:
    raise RuntimeError("找不到 Chapter-7/Mcp，请 cd 到 Mcp 目录或书根目录再打开 notebook")

_mcp_dir = _mcp_dir.resolve()
if str(_mcp_dir) not in sys.path:
    sys.path.insert(0, str(_mcp_dir))

from mcp_paths import bootstrap_paths

bootstrap_paths()

if sys.platform == "win32":
    for _s in (sys.stdout, sys.stderr):
        try:
            _s.reconfigure(encoding="utf-8")
        except Exception:
            pass

print("Mcp 目录:", _mcp_dir)

Mcp 目录: D:\myproject\mira-ai-lab\agent-systems-book\Chapter-7\Mcp


## 1. 课堂示例：用户 Query 与系统提示词

- 用户用自然语言提问。
- 系统提示词要求 Agent **只能**通过 MCP 工具 `recommend_hotel_tool` 查酒店。
- 工具返回候选列表后，由大模型**只推荐一家**最合适的。

In [5]:
USER_QUERY = "我要去大同玩三天给我推荐酒店，需要近景区"

SYSTEM_INSTRUCTION = """你是酒店推荐助手，只能通过 MCP 工具 recommend_hotel_tool 查酒店。
规则：
1. 从用户话里提取 city（必填）；预算、区域、品牌等写入工具参数，工具会返回候选列表。
2. 根据工具返回的 hotels 列表，结合用户预算与偏好，由你挑选并只向用户推荐一家最合适的酒店。
3. 非酒店相关问题，回复：我只能协助酒店推荐。"""

print("用户:", USER_QUERY)

用户: 我要去大同玩三天给我推荐酒店，需要近景区


## 2. 检查 MCP 连接

先确认 Server 已启动；若失败，请回到终端运行 `python hotel_mcp_server.py`。

In [6]:
from hotel_mcp_client import sse_url

print("MCP SSE 地址:", sse_url())

MCP SSE 地址: http://127.0.0.1:8765/sse


## 3. 【对比】直接调用 MCP 工具（不经过 Agent）

程序员显式调用 `recommend_hotel_tool`，拿到 JSON 酒店列表。  
这一步说明：**工具能力在 MCP Server 里**，Client 只是远程 RPC。

In [7]:
import json

from hotel_mcp_client import fetch_hotels_via_mcp

result = await fetch_hotels_via_mcp("大同")

if not result:
    raise RuntimeError(
        "MCP 调用失败，请先启动: python hotel_mcp_server.py\n"
        f"SSE: {sse_url()}"
    )

print(f"搜索词: {result.get('search_query')}")
print(f"数据源: {result.get('data_source')}")
print(f"共 {len(result.get('hotels') or [])} 家候选\n")

for i, h in enumerate(result.get("hotels") or [], 1):
    print(f"  {i}. {h.get('name')} | {h.get('district') or h.get('address')}")

搜索词: 酒店
数据源: baidu_place_v2
共 10 家候选

  1. 天镇久天宾馆 | 天镇县
  2. 佳园宾馆 | 云冈区
  3. 大同海波诚信驿站民宿 | 平城区
  4. 子鼠丑牛客栈(大同古城店) | 平城区
  5. 家馨宾馆 | 广灵县
  6. 大同朴宿微澜民宿 | 平城区
  7. 大同碧海情缘公寓 | 平城区
  8. 微风轻语民宿 | 平城区
  9. 闲然之家民宿 | 平城区
  10. 玥庭兰舍民宿 | 浑源县


## 4. 从 MCP Server 加载工具 → LangChain Tool

`MultiServerMCPClient.get_tools()` 会：
1. 通过 SSE 连接 Server；
2. 拉取 `tools/list`；
3. 转成 LangChain 可用的 `BaseTool` 列表。

本课 Agent 主要用 **`recommend_hotel_tool`**（查列表）；`hotel_agent_query` 是「整句自然语言 → 服务端 Agent」的封装，此处仅展示列表。

In [8]:
from typing import List

from langchain_core.tools import BaseTool
from langchain_mcp_adapters.client import MultiServerMCPClient

from hotel_mcp_client import sse_url


def mcp_client() -> MultiServerMCPClient:
    return MultiServerMCPClient(
        {
            "hotel": {
                "transport": "sse",
                "url": sse_url(),
                "timeout": 10.0,
                "sse_read_timeout": 300.0,
            }
        }
    )


async def load_mcp_tools() -> List[BaseTool]:
    client = mcp_client()
    tools = await client.get_tools()
    if not tools:
        raise RuntimeError("未获取到 MCP 工具，请确认 hotel_mcp_server.py 已启动")
    return tools


mcp_tools = await load_mcp_tools()
print("已加载 MCP 工具:", [t.name for t in mcp_tools])
print("\nrecommend_hotel_tool 描述:")
for t in mcp_tools:
    if t.name == "recommend_hotel_tool":
        print(t.description)

已加载 MCP 工具: ['recommend_hotel_tool', 'hotel_agent_query']

recommend_hotel_tool 描述:
查询酒店候选列表（百度/高德 POI），返回 hotels 数组。


## 5. 组装 Agent（大模型 + MCP 工具 + 系统提示词）

与 Chapter-5 相同：用 `create_agent`；区别是 `tools=` 来自 MCP，而非本地 `@tool`。

In [9]:
import os

import httpx
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver


def create_llm() -> ChatOpenAI:
    api_key = os.getenv("DASHSCOPE_API_KEY") or os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("请设置 DASHSCOPE_API_KEY")
    ssl_verify = os.getenv("OPENAI_SSL_VERIFY", "false").lower() not in ("0", "false", "no")
    return ChatOpenAI(
        model=os.getenv("DASHSCOPE_CHAT_MODEL", "qwen-plus"),
        temperature=0,
        api_key=api_key,
        base_url=os.getenv(
            "DASHSCOPE_CHAT_BASE_URL",
            "https://dashscope.aliyuncs.com/compatible-mode/v1",
        ).rstrip("/"),
        http_client=httpx.Client(verify=ssl_verify),
    )


# 教学演示：只绑定查列表工具，避免 Agent 误调 hotel_agent_query
agent_tools = [t for t in mcp_tools if t.name == "recommend_hotel_tool"]

agent = create_agent(
    create_llm(),
    tools=agent_tools,
    system_prompt=SYSTEM_INSTRUCTION,
    checkpointer=MemorySaver(),
)

print("Agent 已就绪，绑定工具:", [t.name for t in agent_tools])

Agent 已就绪，绑定工具: ['recommend_hotel_tool']


## 6. 【正文】Agent 自动调用 MCP 工具（流式观察）

运行下面单元格，按顺序观察：
1. `>>> MCP 工具调用` — 模型提取的 city / preferences
2. `>>> 工具返回` — MCP Server 返回的 hotels JSON
3. `模型继续:` — 模型读列表后给出的**最终推荐**

> 流式事件名必须是 **`on_chat_model_stream`**（不是 `on_chat_stream`）。

In [10]:
from typing import Any


def short_json(data: Any) -> str:
    if hasattr(data, "content"):
        data = data.content
    if isinstance(data, str):
        try:
            data = json.loads(data)
        except json.JSONDecodeError:
            return data[:800]
    text = json.dumps(data, ensure_ascii=False, indent=2) if isinstance(data, (dict, list)) else str(data)
    return text if len(text) <= 800 else text[:800] + "…"


inputs = {"messages": [("user", USER_QUERY)]}
config = {"configurable": {"thread_id": "notebook_demo_mcp"}}

print(f"用户: {USER_QUERY}\n")
print("模型回答: ", end="", flush=True)

streamed = False
async for event in agent.astream_events(inputs, config, version="v2"):
    kind = event["event"]

    if kind == "on_chat_model_stream":
        chunk = event["data"]["chunk"].content
        if chunk:
            if isinstance(chunk, list):
                chunk = "".join(
                    block.get("text", "") if isinstance(block, dict) else str(block)
                    for block in chunk
                )
            print(chunk, end="", flush=True)
            streamed = True

    elif kind == "on_tool_start":
        print(f"\n\n>>> MCP 工具调用: {event.get('name')}")
        print(f">>> 参数: {short_json(event['data'].get('input'))}")

    elif kind == "on_tool_end":
        print(f">>> 工具返回: {short_json(event['data'].get('output'))}\n")
        print("模型继续: ", end="", flush=True)

    elif kind == "on_chain_end" and event.get("name") == "LangGraph":
        if not streamed:
            out = event.get("data", {}).get("output") or {}
            messages = out.get("messages") if isinstance(out, dict) else None
            if messages:
                last = messages[-1]
                content = getattr(last, "content", None) or ""
                if content:
                    print(content, end="", flush=True)
        break

print()

用户: 我要去大同玩三天给我推荐酒店，需要近景区

模型回答: 

>>> MCP 工具调用: recommend_hotel_tool
>>> 参数: {
  "city": "大同",
  "preferences": "近景区"
}
>>> 工具返回: [
  {
    "type": "text",
    "text": "{\n  \"city\": \"大同\",\n  \"search_query\": \"近景区 酒店\",\n  \"preferences\": \"近景区\",\n  \"budget_cny_per_night_max\": null,\n  \"hotels\": [\n    {\n      \"name\": \"浑源旨岭宜景酒店(北岳恒山景区真武庙店)\",\n      \"district\": \"浑源县\",\n      \"address\": \"恒山景区真武庙旁边\",\n      \"tel\": \"15525220428\",\n      \"location\": \"113.740723,39.668781\",\n      \"rating\": 4.8,\n      \"avg_price_cny\": null,\n      \"type\": \"hotel\"\n    },\n    {\n      \"name\": \"遇见·恒山\",\n      \"district\": \"浑源县\",\n      \"address\": \"大同市恒山北路国际绿洲·和园七号楼一单元801\",\n      \"tel\": \"13935273488\",\n      \"location\": \"113.70018,39.71848\",\n      \"rating\": 4.8,\n      \"avg_price_cny\": null,\n      \"type\": \"hotel\"\n    },\n    {\n      \"name\": \"瑞福民宿(大同古城景区店)\",\n      \…

模型继续: 为您推荐：**瑞福民宿（大同古城景区店）**

理由：  
- 地处大同核心景区——**大同古城内**（云路街20号），步

## 7. 练习（可选）

修改 `USER_QUERY` 后重新运行 **第 6 节** 单元格，例如：

- `"上海出差两晚，预算 500 以内，靠近外滩"`
- `"杭州西湖边安静一点的酒店"`

观察模型传给 `recommend_hotel_tool` 的参数是否变化。

In [12]:
# 练习：改这里后再跑上一格
USER_QUERY = "上海出差两晚，预算 500 以内，靠近外滩"

## 8. 小结

| 环节 | Chapter-5（本地工具） | 本课（MCP） |
|------|----------------------|-------------|
| 工具定义 | notebook / py 里 `@tool` | `hotel_mcp_server.py` 里 `@mcp.tool` |
| Agent 绑定 | `tools=[recommend_hotel]` | `tools=await load_mcp_tools()` |
| 查酒店 API | demo 进程内直接调 | MCP Server 进程内调 |
| 部署 | 单体脚本 | Server 可独立部署、多 Client 共享 |

**一条链路**：用户提问 → Agent → **MCP `recommend_hotel_tool`** → 百度地图 → 模型选一家回复。